# Verification of Lemma 3.3 via Gröbner basis elimination (SageMath)

This notebook verifies the elimination step and the explicit polynomial certificate used in Lemma 3.3.


## 1. Define the Polynomial Ring and Variables

In [ ]:
# Define the polynomial ring with lexicographic order r > s > a > b
R.<r, s, a, b> = PolynomialRing(QQ, order='lex')
print("Polynomial ring defined successfully:")
print(R)

## 2. Define $L_i$ and $R_i$

In [ ]:
# Define $L_i$ and $R_i$
L1 = a*(b^2 + b - a + s + s*b) - b*(a^2 + s + r*a)
R1 = a*(b - a + s) - b*r

L2 = a*(r^2 + r + s + r*a + s*b - a) - r*(a^2 + s + r*a)
R2 = a*(s + r*a + r - a) - r^2

L3 = a*(s^2 + 2*s + r*a + s*b - a) - s*(a^2 + s + r*a)
R3 = a*(r + s*b + s - a) - s*r

print("Defined $L_i$ and $R_i$:")
print(f"L1 = {L1}")
print(f"R1 = {R1}")
print(f"\nL2 = {L2}")
print(f"R2 = {R2}")
print(f"\nL3 = {L3}")
print(f"R3 = {R3}")

## 3. Define $E_i = (b-a)(L_i - R_i) - R_i(s-r)$

In [ ]:
# Define $E_i = (b-a)(L_i-R_i) - R_i(s-r)$
E1 = (b - a)*(L1 - R1) - R1*(s - r)
E2 = (b - a)*(L2 - R2) - R2*(s - r)
E3 = (b - a)*(L3 - R3) - R3*(s - r)

print("="*70)
print("Defined polynomials E1, E2, and E3:")
print("="*70)
print(f"\nE1 = {E1}")
print(f"\nE2 = {E2}")
print(f"\nE3 = {E3}")

## 4. Compute the Gröbner Basis

In [ ]:
print("="*70)
print("Computing the Grobner basis (lex order: r > s > a > b)...")
print("This may take some time...")
print("="*70)

# Define the ideal
I = R.ideal([E1, E2, E3])

# Compute the Gröbner Basis
G = I.groebner_basis()

print(f"\nThe Grobner basis contains {len(G)} polynomials")
print("\nFull Grobner basis:")
for i, g in enumerate(G, 1):
    print(f"\nBasis element {i}: {g}")

## 5. Extract Polynomials in $a, b$ Only

In [ ]:
# Extract the polynomials involving only a and b from the Grobner basis
basis_ab = []
for g in G:
    # Check whether the polynomial involves only a and b (not r or s)
    vars_in_g = g.variables()
    if all(v in [a, b] for v in vars_in_g):
        basis_ab.append(g)

print("="*70)
print("Polynomials in a and b obtained by elimination:")
print("="*70)

if basis_ab:
    for i, p in enumerate(basis_ab, 1):
        print(f"\nPolynomial {i}:")
        print(p)
        print(f"\nFactorization:")
        print(factor(p))
else:
    print("\nNo polynomial involving only a and b was found!")

## 6. Verify the $Q(a,b)$ Factor

In [ ]:
# Check whether a polynomial involving only a and b has been found
try:
    if len(basis_ab) > 0:
        print("="*70)
        print("Extracting the $Q(a,b)$ factor:")
        print("="*70)
        
        poly = basis_ab[0]
        factored = factor(poly)
        
        print(f"\nComplete factorized form:")
        print(factored)
        
        print(f"\nStandard form: a^2*b*(a-b)^3*Q(a,b)")
        print(f"\nHere Q(a,b) should be:")
        print(f"a³b + a²b² - 2a²b - a² + ab³ - 2ab² + 2ab - b²")
        
        # Define the expected $Q(a,b)$
        Q_expected = a^3*b + a^2*b^2 - 2*a^2*b - a^2 + a*b^3 - 2*a*b^2 + 2*a*b - b^2
        print(f"\nVerifying Q(a,b) = {Q_expected}")
        
        # Attempt to extract Q from the factorization
        # The polynomial should equal a^2 * b * (a-b)^3 * Q
        expected_full = a^2 * b * (a-b)^3 * Q_expected
        print(f"\nChecking whether the full polynomial matches:")
        print(f"poly == expected_full: {poly == expected_full}")
    else:
        print("Error: no polynomial involving only a and b was found. Please run the previous cells first!")
except NameError:
    print("Error: variable basis_ab is undefined. Please run Cell 5 first!")

## 7. Compute a coefficient certificate automatically
SageMath's `lift` command computes coefficients $H_i^{\rm auto}$ satisfying $P=\sum_i H_i^{\rm auto}E_i$.

In [ ]:
E = [E1, E2, E3]
Q = a*b*((a - 1)^2 + (b - 1)^2) + a^2*b^2 - a^2 - b^2
P = a^2*b*(a - b)^3*Q

H_auto = P.lift(E)
certificate_auto = sum(H_auto[i]*E[i] for i in range(3))
assert certificate_auto == P

print('The automatically computed coefficients are stored in H_auto.')
for i, H in enumerate(H_auto, 1):
    print(f'H{i}_auto: degree {H.total_degree()}, {len(H.monomials())} monomials')
print('Verified: sum(H_auto[i]*E[i]) = P.')

## 8. Verify the compact certificate used in the paper
Ideal-membership representations are not unique. The paper uses the following more compact representative; its difference from `H_auto` is verified to be a syzygy of $(E_1,E_2,E_3)$.

In [ ]:
t = s - r
D = a^2 + a*b + b^2 - a - b
C = D - a - b + 2
B = (a + b)*D + a*b*C

H1 = (a - b)*(a*Q + r*(a*B - (a + 1)*(Q + D) - r*C)) \
     + t*(b*r*C + a*(D - B) - t*D)
H2 = t*((r + t)*D - B) \
     - (a - b)*b*(a*B - (a + 1)*(Q + D) - (r + t)*C)
H3 = t*(B - r*D) \
     + (a - b)*a*(b*B - (b + 1)*(Q + D) - r*C)

H_paper = [H1, H2, H3]
certificate_paper = sum(H_paper[i]*E[i] for i in range(3))
assert all(H in R for H in (H1, H2, H3))
assert certificate_paper == P

difference = vector(R, H_auto) - vector(R, H_paper)
assert sum(difference[i]*E[i] for i in range(3)) == 0

for name, H in [('H1', H1), ('H2', H2), ('H3', H3)]:
    print(f'{name} = {H}')
print('Verified: H1*E1 + H2*E2 + H3*E3 = a^2*b*(a-b)^3*Q(a,b).')
print('Verified: H_auto - H_paper is a syzygy of (E1,E2,E3).')

## 9. Summary

In [ ]:
print("="*70)
print("Verification completed")
print("="*70)
print("\nConclusion:")
print("A generator of the elimination ideal I ∩ Q[a,b] is:")
print("P(a,b) = a²·b·(a-b)³·Q(a,b)")
print("\nHere Q(a,b) = a^3*b + a^2*b^2 - 2*a^2*b - a^2 + a*b^3 - 2*a*b^2 + 2*a*b - b^2")
print("\nUnder the constraints a, b > 0 and a ≠ b, since a^2*b*(a-b)^3 ≠ 0,")
print("therefore P(a,b) = 0 if and only if Q(a,b) = 0")